# **Data Preprocessing**

## **1. Training Dataset**

### **1.1. Select random 1500 samples from original training data**

In [3]:
import os
import shutil
import pandas as pd

# Paths
excel_file = "InfantFaceGeneration_Dataset/Baby_emotion_1500_samples.xlsx"          
source_folder = "InfantFaceGeneration_Dataset/orig_train"         # Folder containing the files
destination_folder = "InfantFaceGeneration_Dataset/orig_train_samples"   # Folder where files will be copied

# Create destination folder if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Read the Excel file
df = pd.read_excel(excel_file)

# Get filenames from the first column
filenames = df.iloc[:, 0].astype(str)

copied = 0
missing = []

for filename in filenames:
    src = os.path.join(source_folder, filename)
    dst = os.path.join(destination_folder, filename)

    if os.path.isfile(src):
        shutil.copy2(src, dst)  
        copied += 1
    else:
        missing.append(filename)

print(f"Copied {copied} files.")

if missing:
    print(f"\nMissing files ({len(missing)}):")
    for f in missing:
        print(f)

Copied 1500 files.


### **1.2. Use Roboflow to crop infant faces only**

In [4]:
import os
import cv2
from inference_sdk import InferenceHTTPClient

# Initialize client
CLIENT = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key="wdOiqsOhfa9jtEgGLfOW"
)

input_folder = "InfantFaceGeneration_Dataset/orig_train_samples"
output_folder = "InfantFaceGeneration_Dataset/orig_train_samples_cropped"

os.makedirs(output_folder, exist_ok=True)

extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for filename in os.listdir(input_folder):

    if not filename.lower().endswith(extensions):
        continue

    image_path = os.path.join(input_folder, filename)

    img = cv2.imread(image_path)

    if img is None:
        print(f"Cannot read {filename}")
        continue

    try:
        result = CLIENT.infer(
            image_path,
            model_id="smart-baby-monitoring-system-y10/3"
        )

        predictions = result.get("predictions", [])

        if len(predictions) == 0:
            print(f"No baby detected: {filename}")
            continue

        # Highest confidence detection
        pred = max(predictions, key=lambda p: p["confidence"])

        x = pred["x"]
        y = pred["y"]
        w = pred["width"]
        h = pred["height"]

        # Convert center coordinates to corners
        x1 = int(x - w / 2)
        y1 = int(y - h / 2)
        x2 = int(x + w / 2)
        y2 = int(y + h / 2)

        # Clamp to image bounds
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img.shape[1], x2)
        y2 = min(img.shape[0], y2)

        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            print(f"Empty crop: {filename}")
            continue

        # Resize (optional)
        crop = cv2.resize(crop, (512, 512))

        output_path = os.path.join(output_folder, filename)
        cv2.imwrite(output_path, crop)

        print(f"Saved: {filename}")

    except Exception as e:
        print(f"Error processing {filename}: {e}")

print("Finished!")

Saved: 01_asian-crying-baby_jpg.rf.oFWKn9QBCUOb8e2kW3W5.jpg
Saved: 04620123_wide_jpg.rf.q4R3SMZuazVPGD96hHIO.jpg
Saved: 08bbe62deda7ed0cf7863950a6433d1f_jpg.rf.IbUJnL02ksLThRzYTO6D.jpg
Saved: 10-month-old-baby-girl-crying-angrily-while-sitting-on-the-g_jpg.rf.Q3xVkBuYcTbC989H6tFx.jpg
Saved: 1000_F_218239836_SBw7FCh5SOmooR2lXXXTTfbaGJtKOYPh_jpg.rf.xaogohn7ybVJf7e1v2Xg.jpg
Saved: 1000_F_23715992_B89y6MnAhGteHvY32cqDJXxfsZq0tQ9v_jpg.rf.ycQVLX4LWY28X9N3wqK8.jpg
Saved: 1000_F_271366689_6I14JSXGgDjZOVmW8V3BS1dNUarGBui7_jpg.rf.mOoRmDGLc6ug0AnmLmL7.jpg
Saved: 1000_F_31421092_7oNFqcjREsfg8x7BW53FhGFz7ACBR8ZP (1)_jpg.rf.ly1I2ARFSRF1OYgsGjDA.jpg
Saved: 109803809-newborn-baby-boy-sad-crying_jpg.rf.7bHTIqPJ4hXHH3CDDf9s.jpg
Saved: 110e3574_jpg.rf.CLpWuco6pINaH6admXYN.jpg
Saved: 112aff2bb8e86b2ab0e0774a40216330_jpg.rf.iUyUO3yIMlHiso1idL7r.jpg
Saved: 134340368-asian-child-baby-sad-cry-face-is-sadness-unhappy_jpg.rf.J7l3KyW9Ep1hBZh8g8LT.jpg
Saved: 147284798_wide_jpg.rf.Vcjx4LCUFej20effYdlQ.jpg
Saved: 1

### **1.3. Verify sample count**

In [9]:
import os
import pandas as pd

# Paths
excel_file = "InfantFaceGeneration_Dataset/Baby_emotion_1500_samples.xlsx"
folder = "InfantFaceGeneration_Dataset/orig_train_samples_cropped"

# Read the Excel file
df = pd.read_excel(excel_file)

# Read all filenames in the folder into a set for fast lookup
folder_files = {
    f for f in os.listdir(folder)
    if os.path.isfile(os.path.join(folder, f))
}

# First column contains the filenames
filenames = df.iloc[:, 0].astype(str)

# Add a 4th column: 'Y' if found, 'N' if not found
df["Found"] = filenames.apply(lambda x: "Y" if x in folder_files else "N")

# Save to a new Excel file
output_excel = "InfantFaceGeneration_Dataset/Baby_emotion_samples_found.xlsx"
df.to_excel(output_excel, index=False)

print(f"Updated Excel saved as: {output_excel}")
print(f"Found: {(df['Found'] == 'Y').sum()}")
print(f"Not Found: {(df['Found'] == 'N').sum()}")

Updated Excel saved as: InfantFaceGeneration_Dataset/Baby_emotion_samples_found.xlsx
Found: 1443
Not Found: 57


### **1.4. Select random 1200 samples(400 each for angry, cry, and happy) from cropped images**

In [11]:
import os
import pandas as pd

# Paths
excel_file = "InfantFaceGeneration_Dataset/final_train_1200_samples.xlsx"
folder = "InfantFaceGeneration_Dataset/orig_train_samples_cropped"

# Read the Excel file
df = pd.read_excel(excel_file)

# Filenames from the first column of the Excel file
excel_files = set(df.iloc[:, 0].astype(str))

deleted = 0
kept = 0

# Iterate through all files in the folder
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)

    # Skip directories
    if not os.path.isfile(file_path):
        continue

    # Delete files not listed in the Excel file
    if filename not in excel_files:
        os.remove(file_path)
        deleted += 1
        print(f"Deleted: {filename}")
    else:
        kept += 1

print("\nDone!")
print(f"Kept: {kept} files")
print(f"Deleted: {deleted} files")

Deleted: 112aff2bb8e86b2ab0e0774a40216330_jpg.rf.iUyUO3yIMlHiso1idL7r.jpg
Deleted: 147284798_wide_jpg.rf.Vcjx4LCUFej20effYdlQ.jpg
Deleted: 15751aaa1fdf194a631dc52227a29d89_jpg.rf.VGbNvkI0uSN36mWv3WFx.jpg
Deleted: 17c97814e24288c51a8fe3cf4598c450_jpg.rf.Vkli7jKGFjX5FvR1PbvK.jpg
Deleted: 1940s-crying-african-american-baby-boy-n215-har001-hars-copy_jpg.rf.UDzrs79IjuGWOoNFjoyu.jpg
Deleted: 240_F_1018962311_y6GeXFeq0b831hSzC2fWLVFtkt93ko5Z_jpg.rf.Y32B544HeAcW7ECsrx55.jpg
Deleted: 240_F_109869277_HHCsRHigWjKxs3a6TeI0oSJmGF6GoBIw_jpg.rf.J50esuUPFbYpw1VTrAZo.jpg
Deleted: 240_F_111101551_sf5uaDAp0VGe9lBYJSfg0fTXECx0tdob_jpg.rf.ij72a6ouiqFGtv9mnzg1.jpg
Deleted: 240_F_258071296_m7WBqNiwjD1bQU32JpxYYOj3gGrHaloC_jpg.rf.f4aFLbeDL9ZYwvfK9Qcw.jpg
Deleted: 240_F_50811503_2TkFNajrBLzTNdLq8V2IEUvkgLegkc7d_jpg.rf.rt1bk4wL8fCJUVOSaNVQ.jpg
Deleted: 240_F_781995011_Xw42T8OqezjFZih82bSHWa6JOIteQJnM_jpg.rf.IYqbSjp9YbW12cUBh9Xe.jpg
Deleted: 360_F_1006859514_P7wgVhlYPL2DC4hrlIYJGynfJ7ciaKE8_jpg.rf.aMXSaOVT2R7L2p

### **1.6. Copy files to final training data directory**

In [13]:
import shutil

shutil.copytree("InfantFaceGeneration_Dataset/orig_train_samples_cropped", "InfantFaceGeneration_Dataset/final_train_samples")

'InfantFaceGeneration_Dataset/final_train_samples'

### **1.7. Create json file for labels**

In [14]:
import json
import os
import pandas as pd

# ----------------------------
# Configuration
# ----------------------------
excel_file = "InfantFaceGeneration_Dataset/final_train_1200_samples.xlsx"
image_folder = "InfantFaceGeneration_Dataset/final_train_samples"
output_json = "InfantFaceGeneration_Dataset/final_train_samples/train_samples.json"

# ----------------------------
# Read Excel
# ----------------------------
df = pd.read_excel(excel_file)

label_columns = ["Angry", "Cry", "Happy"]

labels = []

for _, row in df.iterrows():

    filename = str(row["filename"])

    # Skip if image doesn't exist
    if not os.path.isfile(os.path.join(image_folder, filename)):
        print(f"Missing: {filename}")
        continue

    # Determine class
    class_id = row[label_columns].astype(int).values.argmax()

    labels.append([filename, int(class_id)])

dataset = {
    "labels": labels
}

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4)

print(f"Saved {output_json}")

Saved InfantFaceGeneration_Dataset/final_train_samples/train_samples.json


### **1.8. Verify json vs files**

In [26]:
import os
import json

# ============================================================
# Paths
# ============================================================
folder = "InfantFaceGeneration_Dataset/final_train_samples"
json_file = os.path.join(folder, "train_samples.json")

# ============================================================
# Read JSON
# ============================================================
with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)

labels = data["labels"]

# ============================================================
# Count labels
# ============================================================
label_counts = {
    0: 0,  # Angry
    1: 0,  # Cry
    2: 0   # Happy
}

for filename, label in labels:
    if label in label_counts:
        label_counts[label] += 1

# ============================================================
# Count actual image files
# ============================================================
extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_files = [
    f for f in os.listdir(folder)
    if f.lower().endswith(extensions)
]

# ============================================================
# Print results
# ============================================================
print("=" * 50)
print("FINAL TRAIN DATASET VERIFICATION")
print("=" * 50)

print(f"Total images in folder: {len(image_files)}")
print(f"Total labels in JSON:   {len(labels)}")

print()
print("Emotion counts from JSON:")
print(f"Angry: {label_counts[0]}")
print(f"Cry:   {label_counts[1]}")
print(f"Happy: {label_counts[2]}")

print()
print(f"Expected total: 1200")
print(f"Actual total:   {len(image_files)}")

print("=" * 50)

FINAL TRAIN DATASET VERIFICATION
Total images in folder: 1200
Total labels in JSON:   1200

Emotion counts from JSON:
Angry: 400
Cry:   400
Happy: 400

Expected total: 1200
Actual total:   1200


### **1.9. Verify image resolution of files**

In [15]:
from PIL import Image
from pathlib import Path
from collections import Counter

image_dir = Path("InfantFaceGeneration_Dataset/final_train_samples")

extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

sizes = Counter()

for img_path in image_dir.rglob("*"):
    if img_path.suffix.lower() in extensions:
        try:
            with Image.open(img_path) as img:
                sizes[img.size] += 1  # (width, height)
        except Exception:
            print(f"Could not open {img_path}")

print("Image resolutions:")
for size, count in sorted(sizes.items()):
    print(f"{size}: {count} images")

Image resolutions:
(512, 512): 1200 images


## **2. Test Dataset**

### **2.1 Use rest of original training files for test**

In [18]:
import os
import shutil
import pandas as pd

# ============================================================
# Paths
# ============================================================
selected_excel = "InfantFaceGeneration_Dataset/Baby_emotion_1500_samples.xlsx"
master_csv = "InfantFaceGeneration_Dataset/orig_train/_classes.csv"

source_folder = "InfantFaceGeneration_Dataset/orig_train"
test_folder = "InfantFaceGeneration_Dataset/orig_test"

output_excel = "InfantFaceGeneration_Dataset/Baby_emotion_rest_samples.xlsx"


# ============================================================
# Create destination folder
# ============================================================
os.makedirs(test_folder, exist_ok=True)


# ============================================================
# Read the 1500-sample Excel
# ============================================================
selected_df = pd.read_excel(selected_excel)

if "filename" not in selected_df.columns:
    raise ValueError(
        f"'filename' column not found in Excel.\n"
        f"Columns found: {list(selected_df.columns)}"
    )

selected_filenames = set(
    selected_df["filename"]
    .astype(str)
    .str.strip()
    .apply(os.path.basename)
)


# ============================================================
# Read master CSV manually
#
# Expected format:
# filename,Angry,Cry,Happy
# ============================================================
valid_rows = []
skipped_rows = []

with open(master_csv, "r", encoding="utf-8") as f:

    # Read header
    header = f.readline().strip().split(",")

    print("CSV Header:", header)

    # Verify header
    expected_header = ["filename", "Angry", "Cry", "Happy"]

    if header != expected_header:
        print("WARNING: Header is different from expected.")
        print("Expected:", expected_header)
        print("Found:   ", header)

    # Read remaining rows
    for line_number, line in enumerate(f, start=2):

        line = line.rstrip("\n\r")

        parts = line.split(",")

        # ----------------------------------------------------
        # Skip malformed rows
        # ----------------------------------------------------
        if len(parts) != 4:

            filename_part = parts[0].strip() if parts else ""

            skipped_rows.append({
                "line": line_number,
                "filename": filename_part,
                "reason": f"Expected 4 fields, found {len(parts)}"
            })

            continue

        filename = parts[0].strip()
        angry = parts[1].strip()
        cry = parts[2].strip()
        happy = parts[3].strip()

        valid_rows.append({
            "filename": filename,
            "Angry": angry,
            "Cry": cry,
            "Happy": happy
        })


# ============================================================
# Create master DataFrame
# ============================================================
master_df = pd.DataFrame(
    valid_rows,
    columns=["filename", "Angry", "Cry", "Happy"]
)


print()
print("=" * 60)
print("MASTER CSV")
print("=" * 60)
print(f"Valid CSV rows:    {len(master_df)}")
print(f"Skipped CSV rows:  {len(skipped_rows)}")


# ============================================================
# Get actual image files in orig_train
# ============================================================
source_files = {
    f
    for f in os.listdir(source_folder)
    if os.path.isfile(os.path.join(source_folder, f))
    and f.lower().endswith(
        (".jpg", ".jpeg", ".png", ".webp")
    )
}


# ============================================================
# Find files NOT in the 1500-sample Excel
# ============================================================
remaining_files = sorted(
    source_files - selected_filenames
)


print()
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Images in orig_train:       {len(source_files)}")
print(f"Images in 1500 Excel:       {len(selected_filenames)}")
print(f"Remaining images:           {len(remaining_files)}")


# ============================================================
# Copy remaining images to orig_test
# ============================================================
copied_count = 0
already_exists = 0

for filename in remaining_files:

    source_path = os.path.join(source_folder, filename)
    destination_path = os.path.join(test_folder, filename)

    if os.path.exists(destination_path):
        already_exists += 1
        continue

    shutil.copy2(
        source_path,
        destination_path
    )

    copied_count += 1


print(f"Images copied:              {copied_count}")
print(f"Already existed:            {already_exists}")


# ============================================================
# Get labels for remaining images
# ============================================================
remaining_df = master_df[
    master_df["filename"].isin(remaining_files)
].copy()


# ============================================================
# Save remaining samples to Excel
# ============================================================
remaining_df.to_excel(
    output_excel,
    index=False
)


print(f"Rows in output Excel:       {len(remaining_df)}")
print(f"Output Excel:               {output_excel}")


# ============================================================
# Emotion counts
# ============================================================
print()
print("=" * 60)
print("EMOTION COUNTS IN TEST SET")
print("=" * 60)

for column in ["Angry", "Cry", "Happy"]:

    count = pd.to_numeric(
        remaining_df[column],
        errors="coerce"
    ).fillna(0).sum()

    print(f"{column:10s}: {int(count)}")


# ============================================================
# Skipped rows
# ============================================================
print()
print("=" * 60)
print("SKIPPED CSV ROWS")
print("=" * 60)

for item in skipped_rows:

    print(
        f"Line {item['line']}: "
        f"{item['reason']}"
    )

print("=" * 60)

CSV Header: ['filename', 'Angry', 'Cry', 'Happy']

MASTER CSV
Valid CSV rows:    2391
Skipped CSV rows:  3

DATASET SUMMARY
Images in orig_train:       2330
Images in 1500 Excel:       1500
Remaining images:           868
Images copied:              868
Already existed:            0
Rows in output Excel:       865
Output Excel:               InfantFaceGeneration_Dataset/Baby_emotion_rest_samples.xlsx

EMOTION COUNTS IN TEST SET
Angry     : 299
Cry       : 286
Happy     : 280

SKIPPED CSV ROWS
Line 204: Expected 4 fields, found 5
Line 715: Expected 4 fields, found 9
Line 728: Expected 4 fields, found 5


### **2.2. Use Roboflow to crop infant faces only**

In [20]:
import os
import cv2
import pandas as pd
from inference_sdk import InferenceHTTPClient

# ============================================================
# Initialize Roboflow client
# ============================================================
CLIENT = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key="wdOiqsOhfa9jtEgGLfOW"
)

# ============================================================
# Paths
# ============================================================
input_folder = "InfantFaceGeneration_Dataset/orig_test"

output_folder = (
    "InfantFaceGeneration_Dataset/"
    "orig_test_samples_cropped"
)

excel_file = (
    "InfantFaceGeneration_Dataset/"
    "Baby_emotion_rest_samples.xlsx"
)

os.makedirs(output_folder, exist_ok=True)

extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

# ============================================================
# Read Excel
# ============================================================
df = pd.read_excel(excel_file)

if "filename" not in df.columns:
    raise ValueError(
        f"'filename' column not found in Excel. "
        f"Columns: {list(df.columns)}"
    )

# Normalize filenames for matching
df["filename"] = (
    df["filename"]
    .astype(str)
    .str.strip()
    .apply(os.path.basename)
)

# ============================================================
# Track files with no baby detected
# ============================================================
files_no_baby = set()

# ============================================================
# Process images
# ============================================================
for filename in os.listdir(input_folder):

    if not filename.lower().endswith(extensions):
        continue

    image_path = os.path.join(input_folder, filename)

    img = cv2.imread(image_path)

    if img is None:
        files_no_baby.add(filename)
        continue

    try:

        result = CLIENT.infer(
            image_path,
            model_id="smart-baby-monitoring-system-y10/3"
        )

        predictions = result.get("predictions", [])

        # ----------------------------------------------------
        # No baby detected
        # ----------------------------------------------------
        if len(predictions) == 0:
            files_no_baby.add(filename)
            continue

        # ----------------------------------------------------
        # Highest confidence detection
        # ----------------------------------------------------
        pred = max(
            predictions,
            key=lambda p: p["confidence"]
        )

        x = pred["x"]
        y = pred["y"]
        w = pred["width"]
        h = pred["height"]

        # ----------------------------------------------------
        # Convert center coordinates to corners
        # ----------------------------------------------------
        x1 = int(x - w / 2)
        y1 = int(y - h / 2)

        x2 = int(x + w / 2)
        y2 = int(y + h / 2)

        # ----------------------------------------------------
        # Clamp to image boundaries
        # ----------------------------------------------------
        x1 = max(0, x1)
        y1 = max(0, y1)

        x2 = min(img.shape[1], x2)
        y2 = min(img.shape[0], y2)

        crop = img[y1:y2, x1:x2]

        # ----------------------------------------------------
        # Invalid crop
        # ----------------------------------------------------
        if crop.size == 0:
            files_no_baby.add(filename)
            continue

        # ----------------------------------------------------
        # Resize to 512x512
        # ----------------------------------------------------
        crop = cv2.resize(
            crop,
            (512, 512)
        )

        # ----------------------------------------------------
        # Save cropped image
        # ----------------------------------------------------
        output_path = os.path.join(
            output_folder,
            filename
        )

        cv2.imwrite(
            output_path,
            crop
        )

    except Exception:
        files_no_baby.add(filename)


# ============================================================
# Remove records for files that were not copied
# ============================================================
if files_no_baby:

    df = df[
        ~df["filename"].isin(files_no_baby)
    ].copy()

    df.to_excel(
        excel_file,
        index=False
    )


# ============================================================
# Count final files
# ============================================================
final_files = [
    f
    for f in os.listdir(output_folder)
    if f.lower().endswith(extensions)
]

# ============================================================
# Final output
# ============================================================
print(f"Files not copied (no baby detected): {len(files_no_baby)}")
print(f"Final number of files in folder: {len(final_files)}")

Files not copied (no baby detected): 14
Final number of files in folder: 854


### **2.3. Select random 750 samples(250 each for angry, cry, and happy) from cropped images**

In [21]:
import pandas as pd

# ============================================================
# Paths
# ============================================================
input_excel = (
    "InfantFaceGeneration_Dataset/"
    "Baby_emotion_rest_samples.xlsx"
)

output_excel = (
    "InfantFaceGeneration_Dataset/"
    "final_test_750_samples.xlsx"
)

# ============================================================
# Read Excel
# ============================================================
df = pd.read_excel(input_excel)

# ============================================================
# Verify columns
# ============================================================
required_columns = ["filename", "Angry", "Cry", "Happy"]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Available columns: {list(df.columns)}"
    )

# ============================================================
# Select 250 samples for each emotion
# ============================================================
angry_df = df[df["Angry"] == 1].sample(
    n=250,
    random_state=42
)

cry_df = df[df["Cry"] == 1].sample(
    n=250,
    random_state=42
)

happy_df = df[df["Happy"] == 1].sample(
    n=250,
    random_state=42
)

# ============================================================
# Combine
# ============================================================
final_df = pd.concat(
    [angry_df, cry_df, happy_df],
    ignore_index=True
)

# Shuffle final dataset
final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ============================================================
# Save
# ============================================================
final_df.to_excel(
    output_excel,
    index=False
)

# ============================================================
# Verify
# ============================================================
print("=" * 50)
print("FINAL TEST DATASET")
print("=" * 50)

print(f"Total samples: {len(final_df)}")
print(f"Angry:         {final_df['Angry'].sum()}")
print(f"Cry:           {final_df['Cry'].sum()}")
print(f"Happy:         {final_df['Happy'].sum()}")

print("=" * 50)
print(f"Saved to: {output_excel}")

FINAL TEST DATASET
Total samples: 750
Angry:         250
Cry:           250
Happy:         250
Saved to: InfantFaceGeneration_Dataset/final_test_750_samples.xlsx


### **2.4. Copy files to final test directory**

In [22]:
import os
import shutil
import pandas as pd

# ============================================================
# Paths
# ============================================================
excel_file = (
    "InfantFaceGeneration_Dataset/"
    "final_test_750_samples.xlsx"
)

source_folder = (
    "InfantFaceGeneration_Dataset/"
    "orig_test_samples_cropped"
)

output_folder = (
    "InfantFaceGeneration_Dataset/"
    "final_test_samples"
)

# ============================================================
# Create output folder
# ============================================================
os.makedirs(output_folder, exist_ok=True)

# ============================================================
# Read Excel
# ============================================================
df = pd.read_excel(excel_file)

if "filename" not in df.columns:
    raise ValueError(
        f"'filename' column not found. "
        f"Available columns: {list(df.columns)}"
    )

# ============================================================
# Copy files
# ============================================================
copied_count = 0
missing_count = 0

for filename in df["filename"]:

    filename = os.path.basename(str(filename).strip())

    source_path = os.path.join(source_folder, filename)
    destination_path = os.path.join(output_folder, filename)

    if not os.path.isfile(source_path):
        missing_count += 1
        continue

    shutil.copy2(source_path, destination_path)
    copied_count += 1

# ============================================================
# Final summary
# ============================================================
print(f"Excel records:  {len(df)}")
print(f"Files copied:   {copied_count}")
print(f"Files missing:  {missing_count}")
print(f"Output folder:  {output_folder}")

Excel records:  750
Files copied:   750
Files missing:  0
Output folder:  InfantFaceGeneration_Dataset/final_test_samples


### **2.5. Create json file for labels**

In [23]:
import json
import os
import pandas as pd

# ----------------------------
# Configuration
# ----------------------------
excel_file = "InfantFaceGeneration_Dataset/final_test_750_samples.xlsx"
image_folder = "InfantFaceGeneration_Dataset/final_test_samples"
output_json = "InfantFaceGeneration_Dataset/final_test_samples/test_samples.json"

# ----------------------------
# Read Excel
# ----------------------------
df = pd.read_excel(excel_file)

label_columns = ["Angry", "Cry", "Happy"]

labels = []

for _, row in df.iterrows():

    filename = str(row["filename"])

    # Skip if image doesn't exist
    if not os.path.isfile(os.path.join(image_folder, filename)):
        print(f"Missing: {filename}")
        continue

    # Determine class
    class_id = row[label_columns].astype(int).values.argmax()

    labels.append([filename, int(class_id)])

dataset = {
    "labels": labels
}

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4)

print(f"Saved {output_json}")

Saved InfantFaceGeneration_Dataset/final_test_samples/test_samples.json


### **2.6. Verify json vs files**

In [24]:
import os
import json

# ============================================================
# Paths
# ============================================================
folder = "InfantFaceGeneration_Dataset/final_test_samples"
json_file = os.path.join(folder, "test_samples.json")

# ============================================================
# Read JSON
# ============================================================
with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)

labels = data["labels"]

# ============================================================
# Count labels
# ============================================================
label_counts = {
    0: 0,  # Angry
    1: 0,  # Cry
    2: 0   # Happy
}

for filename, label in labels:
    if label in label_counts:
        label_counts[label] += 1

# ============================================================
# Count actual image files
# ============================================================
extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_files = [
    f for f in os.listdir(folder)
    if f.lower().endswith(extensions)
]

# ============================================================
# Print results
# ============================================================
print("=" * 50)
print("FINAL TEST DATASET VERIFICATION")
print("=" * 50)

print(f"Total images in folder: {len(image_files)}")
print(f"Total labels in JSON:   {len(labels)}")

print()
print("Emotion counts from JSON:")
print(f"Angry: {label_counts[0]}")
print(f"Cry:   {label_counts[1]}")
print(f"Happy: {label_counts[2]}")

print()
print(f"Expected total: 750")
print(f"Actual total:   {len(image_files)}")

print("=" * 50)

FINAL TEST DATASET VERIFICATION
Total images in folder: 750
Total labels in JSON:   750

Emotion counts from JSON:
Angry: 250
Cry:   250
Happy: 250

Expected total: 750
Actual total:   750


### **2.7. Verify image resolution of files**

In [27]:
from PIL import Image
from pathlib import Path
from collections import Counter

image_dir = Path("InfantFaceGeneration_Dataset/final_test_samples")

extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

sizes = Counter()

for img_path in image_dir.rglob("*"):
    if img_path.suffix.lower() in extensions:
        try:
            with Image.open(img_path) as img:
                sizes[img.size] += 1  # (width, height)
        except Exception:
            print(f"Could not open {img_path}")

print("Image resolutions:")
for size, count in sorted(sizes.items()):
    print(f"{size}: {count} images")

Image resolutions:
(512, 512): 750 images
